In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import signal
from scipy.signal import welch
from pathlib import Path
%matplotlib inline
import joblib
import seaborn as sns
sns.set(font_scale=1.2)
from scipy.integrate import simpson


In [3]:
# List All Data
spath = 'eeg-analysis'

p = Path(spath)
dpath = [x for x in p.iterdir() if x.is_file() and x.suffix == '.txt']
dpath = [x.as_posix() for x in dpath]

dpath = sorted(dpath)

In [4]:
# Filter Design

# Bandpass filter
def bandpass(start, stop, data, fs = 125):
    bp_Hz = np.array([start, stop])
    b, a = signal.butter(4, bp_Hz / (fs / 2.0), btype='bandpass')
    return signal.lfilter(b, a, data, axis=0)

# Notch filter
def notch(val, data, fs= 125):
    notch_freq_Hz = np.array([float(val)])
    for freq_Hz in np.nditer(notch_freq_Hz):
        bp_stop_Hz = freq_Hz + 3.0 * np.array([-1, 1])
        b, a = signal.butter(3, bp_stop_Hz / (fs / 2.0), 'bandstop')
        fin = data = signal.lfilter(b, a, data)
    return fin

# FFT Function for checking
def fft(data, fs):
    L = len(data)
    freq = np.linspace(0.0, 1.0 / (2.0 * fs **-1), L // 2)
    yi = np.fft.fft(data)[1:]
    y = yi[range(int(L / 2))]
    return freq, abs(y)

In [5]:
# Load data to pandas
def openbci_to_pandas(path):
    """
    Turn OpenBCI data to Pandas DataFrame.
    Slice channels data only.
    Rename the channels name

    Parameters
    ----------
    path : String
           The file location
    
    Returns
    --------
    df  : DataFrame
    """
    df = pd.read_csv(path, skiprows=4)
    df = df.iloc[:,1:17]
    df.columns = ['Fp1', 'Fp2', 'C3', 'C4', 'T5', 'T6', 'O1', 'O2', 'F7', 'F8', 'F3', 'F4', 'T3', 'T4','P3','P4']

    return df


In [6]:
# Load all data to pandas
all_data = []
for i in dpath:
    df = openbci_to_pandas(i)
    all_data.append(df)

print(len(all_data))

ValueError: Length mismatch: Expected axis has 0 elements, new values have 16 elements

In [ ]:
print(len(all_data))

0


In [ ]:
all_data[6].shape

IndexError: list index out of range

In [ ]:
# fs, notch, band settings
fs_notch = 50
fs = 125

In [ ]:
# Subject 2 Only
s3 = [7, 8, 9, 10]

# notch filter for multiple data
s3_notch = []

for i in s3:
    notch_ch = []
    data = all_data[i]
    
    for j in range(data.shape[1]):
        notch_ch.append(notch(fs_notch,data.iloc[:,j], fs))
    
    notch_ch = np.array(notch_ch)
    s3_notch.append(notch_ch)

In [ ]:
s3_notch[2].shape

In [ ]:
plt.figure(figsize=(15,5))
for i in range(len(s3_notch[0])):
    plt.plot(s3_notch[0][i])

In [ ]:
# check fft of applied notch
for i in range(len(s3_notch[0])):
    freq, y = fft(s3_notch[0][i], fs)
    plt.plot(freq, y)
plt.ylim(0, 1e7)
plt.xlim(0,100)

In [ ]:
#applied bandpass filter = 5-50 and notch = 60
band = (1,50)

s3_bandpass_notch = []

for i in range(len(s3_notch)):
    bandpass_notch_channels = []
    data_notch = s3_notch[i]

    for j in range(len(data_notch)):
        bandpass_notch_channels.append(bandpass(band[0],band[1], data_notch[j], fs))
    
    bandpass_notch_channels = np.array(bandpass_notch_channels)
    s3_bandpass_notch.append(bandpass_notch_channels)

In [ ]:
plt.figure(figsize=(15,5))
for i in range(len(s3_bandpass_notch[0])):
    plt.plot(s3_bandpass_notch[0][i])

In [ ]:
s3_bandpass_notch[0].shape

In [ ]:
# Cut-off begining of the signal
def cut_off(data, begin, duration, fs):
    duration = (duration * fs)

    if begin != 0:
        begin = (begin * fs) - 1
    else:
        begin = 0 + ((begin * fs) - 1)

    cut_data = data[:,begin:begin+duration]
    
    return cut_data

In [ ]:
s3_bandpass_notch_sliced = []

for i in range(len(s3_bandpass_notch)):
    slice_data = cut_off(s3_bandpass_notch[i], 3, 60, fs)
    s3_bandpass_notch_sliced.append(slice_data)

In [ ]:
s3_bandpass_notch_sliced[0].shape

In [ ]:
plt.plot(s3_bandpass_notch_sliced[3][0])

In [ ]:
joblib.dump(s3_bandpass_notch_sliced, 's3_cutoff.pkl')

In [ ]:
s3_bandpass_notch_sliced = joblib.load('s3_cutoff.pkl')

In [ ]:
cols = ["Fp1", "Fp2", "C3", "C4", "T5", "T6", "O1", "O2", "F7", "F8", "F3", "F4", "T3", "T4", "P3", "P4"]

In [ ]:
# Perform bandpass on each signal each state
delta = (1,4)
tetha = (4,8)
alpha = (8,14)
beta = (14,30)
gamma = (30,50)

# Resting Stage

In [ ]:
# Stage 1 - Resting State
delta_stage1 = []
tetha_stage1 = []
alpha_stage1 = []
beta_stage1 = []
gamma_stage1 = []
for i in range(s3_bandpass_notch_sliced[0].shape[0]):
    freqs, psd = welch(s3_bandpass_notch_sliced[0][i], nperseg=125, noverlap=0.5, fs=125)
    
    # Band Range
    idx_delta = np.logical_and(freqs >= delta[0], freqs <= delta[1])
    idx_tetha = np.logical_and(freqs >= tetha[0], freqs <= tetha[1])
    idx_alpha = np.logical_and(freqs >= alpha[0], freqs <= alpha[1])
    idx_beta = np.logical_and(freqs >= beta[0], freqs <= beta[1])
    idx_gamma = np.logical_and(freqs >= gamma[0], freqs <= gamma[1])
    
    # Frequency resolution
    freq_res = freqs[1] - freqs[0]
    
    # Compute the absolute power by approximating the area under the curve
    delta_power = simps(psd[idx_delta], dx=freq_res)
    tetha_power = simps(psd[idx_tetha], dx=freq_res)
    alpha_power = simps(psd[idx_alpha], dx=freq_res)
    beta_power = simps(psd[idx_beta], dx=freq_res)
    gamma_power = simps(psd[idx_gamma], dx=freq_res)
    
    # Relative power
    total_power = simps(psd, dx=freq_res)
    delta_rel_power = delta_power / total_power
    tetha_rel_power = tetha_power / total_power
    alpha_rel_power = alpha_power / total_power
    beta_rel_power = beta_power / total_power
    gamma_rel_power = gamma_power / total_power
    
    delta_stage1.append(delta_rel_power)
    tetha_stage1.append(tetha_rel_power)
    alpha_stage1.append(alpha_rel_power)
    beta_stage1.append(beta_rel_power)
    gamma_stage1.append(gamma_rel_power)

In [ ]:
bands_stage1 = {
    "Delta" : delta_stage1,
    "Tetha" : tetha_stage1,
    "Alpha" : alpha_stage1,
    "Beta" : beta_stage1,
    "Gamma" : gamma_stage1
}

df_bands_stage1 = pd.DataFrame(bands_stage1, index=cols)

In [ ]:
df_bands_stage1.head()

In [ ]:
df_bands_stage1.plot(kind='bar', figsize=(20, 8))
plt.xlabel('Channels')
plt.ylabel('Relatives Power')
plt.show

In [ ]:
# Mean Channles Values
mean_delta_stage1 = np.mean(delta_stage1)
mean_tetha_stage1 = np.mean(tetha_stage1)
mean_alpha_stage1 = np.mean(alpha_stage1)
mean_beta_stage1 = np.mean(beta_stage1)
mean_gamma_stage1 = np.mean(gamma_stage1)

In [ ]:
# Data
categories = ["Delta", "Theta", "Alpha", "Beta", "Gamma"]
values = [mean_delta_stage1, mean_tetha_stage1, mean_alpha_stage1, mean_beta_stage1, mean_gamma_stage1]

# Plot
plt.figure(figsize=(8, 5))
plt.bar(categories, values, color=['blue', 'green', 'orange', 'red', 'purple'])

# Adding labels and title
plt.title("Mean Band Relative Power - Resting State", fontsize=16)
plt.xlabel("EEG Bands", fontsize=12)
plt.ylabel("Relative Power", fontsize=12)
plt.xticks(fontsize=10)
plt.yticks(fontsize=10)
plt.tight_layout()
plt.show()


# Go / NoGo

In [ ]:
# Stage 2 - Go / NoGo
delta_stage2 = []
tetha_stage2 = []
alpha_stage2 = []
beta_stage2 = []
gamma_stage2 = []
for i in range(s3_bandpass_notch_sliced[1].shape[0]):
    freqs, psd = welch(s3_bandpass_notch_sliced[1][i], nperseg=125, noverlap=0.5, fs=125)
    
    # Band Range
    idx_delta = np.logical_and(freqs >= delta[0], freqs <= delta[1])
    idx_tetha = np.logical_and(freqs >= tetha[0], freqs <= tetha[1])
    idx_alpha = np.logical_and(freqs >= alpha[0], freqs <= alpha[1])
    idx_beta = np.logical_and(freqs >= beta[0], freqs <= beta[1])
    idx_gamma = np.logical_and(freqs >= gamma[0], freqs <= gamma[1])
    
    # Frequency resolution
    freq_res = freqs[1] - freqs[0]
    
    # Compute the absolute power by approximating the area under the curve
    delta_power = simps(psd[idx_delta], dx=freq_res)
    tetha_power = simps(psd[idx_tetha], dx=freq_res)
    alpha_power = simps(psd[idx_alpha], dx=freq_res)
    beta_power = simps(psd[idx_beta], dx=freq_res)
    gamma_power = simps(psd[idx_gamma], dx=freq_res)
    
    # Relative power
    total_power = simps(psd, dx=freq_res)
    delta_rel_power = delta_power / total_power
    tetha_rel_power = tetha_power / total_power
    alpha_rel_power = alpha_power / total_power
    beta_rel_power = beta_power / total_power
    gamma_rel_power = gamma_power / total_power
    
    delta_stage2.append(delta_rel_power)
    tetha_stage2.append(tetha_rel_power)
    alpha_stage2.append(alpha_rel_power)
    beta_stage2.append(beta_rel_power)
    gamma_stage2.append(gamma_rel_power)

In [ ]:
bands_stage2 = {
    "Delta" : delta_stage2,
    "Tetha" : tetha_stage2,
    "Alpha" : alpha_stage2,
    "Beta" : beta_stage2,
    "Gamma" : gamma_stage2
}

df_bands_stage2 = pd.DataFrame(bands_stage2, index=cols)

In [ ]:
df_bands_stage2.head()

In [ ]:
df_bands_stage2.plot(kind='bar', figsize=(20, 8))
plt.xlabel('Channels')
plt.ylabel('Relatives Power')
plt.show

In [ ]:
# Mean Channles Values
mean_delta_stage2 = np.mean(delta_stage2)
mean_tetha_stage2 = np.mean(tetha_stage2)
mean_alpha_stage2 = np.mean(alpha_stage2)
mean_beta_stage2 = np.mean(beta_stage2)
mean_gamma_stage2 = np.mean(gamma_stage2)

# Data
categories = ["Delta", "Theta", "Alpha", "Beta", "Gamma"]
values = [mean_delta_stage2, mean_tetha_stage2, mean_alpha_stage2, mean_beta_stage2, mean_gamma_stage2]

# Plot
plt.figure(figsize=(8, 5))
plt.bar(categories, values, color=['blue', 'green', 'orange', 'red', 'purple'])

# Adding labels and title
plt.title("Mean Band Relative Power - Go/NoGo", fontsize=16)
plt.xlabel("EEG Bands", fontsize=12)
plt.ylabel("Relative Power", fontsize=12)
plt.xticks(fontsize=10)
plt.yticks(fontsize=10)
plt.tight_layout()
plt.show()

# Reading

In [ ]:
# Stage 3 - Reading
delta_stage3 = []
tetha_stage3 = []
alpha_stage3 = []
beta_stage3 = []
gamma_stage3 = []
for i in range(s3_bandpass_notch_sliced[2].shape[0]):
    freqs, psd = welch(s3_bandpass_notch_sliced[2][i], nperseg=125, noverlap=0.5, fs=125)
    
    # Band Range
    idx_delta = np.logical_and(freqs >= delta[0], freqs <= delta[1])
    idx_tetha = np.logical_and(freqs >= tetha[0], freqs <= tetha[1])
    idx_alpha = np.logical_and(freqs >= alpha[0], freqs <= alpha[1])
    idx_beta = np.logical_and(freqs >= beta[0], freqs <= beta[1])
    idx_gamma = np.logical_and(freqs >= gamma[0], freqs <= gamma[1])
    
    # Frequency resolution
    freq_res = freqs[1] - freqs[0]
    
    # Compute the absolute power by approximating the area under the curve
    delta_power = simps(psd[idx_delta], dx=freq_res)
    tetha_power = simps(psd[idx_tetha], dx=freq_res)
    alpha_power = simps(psd[idx_alpha], dx=freq_res)
    beta_power = simps(psd[idx_beta], dx=freq_res)
    gamma_power = simps(psd[idx_gamma], dx=freq_res)
    
    # Relative power
    total_power = simps(psd, dx=freq_res)
    delta_rel_power = delta_power / total_power
    tetha_rel_power = tetha_power / total_power
    alpha_rel_power = alpha_power / total_power
    beta_rel_power = beta_power / total_power
    gamma_rel_power = gamma_power / total_power
    
    delta_stage3.append(delta_rel_power)
    tetha_stage3.append(tetha_rel_power)
    alpha_stage3.append(alpha_rel_power)
    beta_stage3.append(beta_rel_power)
    gamma_stage3.append(gamma_rel_power)

In [ ]:
bands_stage3 = {
    "Delta" : delta_stage3,
    "Tetha" : tetha_stage3,
    "Alpha" : alpha_stage3,
    "Beta" : beta_stage3,
    "Gamma" : gamma_stage3
}

df_bands_stage3 = pd.DataFrame(bands_stage3, index=cols)

In [ ]:
df_bands_stage3.head()

In [ ]:
df_bands_stage3.plot(kind='bar', figsize=(20, 8))
plt.xlabel('Channels')
plt.ylabel('Relatives Power')
plt.show

In [ ]:
# Mean Channles Values
mean_delta_stage3 = np.mean(delta_stage3)
mean_tetha_stage3 = np.mean(tetha_stage3)
mean_alpha_stage3 = np.mean(alpha_stage3)
mean_beta_stage3 = np.mean(beta_stage3)
mean_gamma_stage3 = np.mean(gamma_stage3)

# Data
categories = ["Delta", "Theta", "Alpha", "Beta", "Gamma"]
values = [mean_delta_stage3, mean_tetha_stage3, mean_alpha_stage3, mean_beta_stage3, mean_gamma_stage3]

# Plot
plt.figure(figsize=(8, 5))
plt.bar(categories, values, color=['blue', 'green', 'orange', 'red', 'purple'])

# Adding labels and title
plt.title("Mean Band Relative Power - Reading", fontsize=16)
plt.xlabel("EEG Bands", fontsize=12)
plt.ylabel("Relative Power", fontsize=12)
plt.xticks(fontsize=10)
plt.yticks(fontsize=10)
plt.tight_layout()
plt.show()

# VIAT-map

In [ ]:
# Stage 4 - VIAT-map
delta_stage4 = []
tetha_stage4 = []
alpha_stage4 = []
beta_stage4 = []
gamma_stage4 = []
for i in range(s3_bandpass_notch_sliced[3].shape[0]):
    freqs, psd = welch(s3_bandpass_notch_sliced[3][i], nperseg=125, noverlap=0.5, fs=125)
    
    # Band Range
    idx_delta = np.logical_and(freqs >= delta[0], freqs <= delta[1])
    idx_tetha = np.logical_and(freqs >= tetha[0], freqs <= tetha[1])
    idx_alpha = np.logical_and(freqs >= alpha[0], freqs <= alpha[1])
    idx_beta = np.logical_and(freqs >= beta[0], freqs <= beta[1])
    idx_gamma = np.logical_and(freqs >= gamma[0], freqs <= gamma[1])
    
    # Frequency resolution
    freq_res = freqs[1] - freqs[0]
    
    # Compute the absolute power by approximating the area under the curve
    delta_power = simps(psd[idx_delta], dx=freq_res)
    tetha_power = simps(psd[idx_tetha], dx=freq_res)
    alpha_power = simps(psd[idx_alpha], dx=freq_res)
    beta_power = simps(psd[idx_beta], dx=freq_res)
    gamma_power = simps(psd[idx_gamma], dx=freq_res)
    
    # Relative power
    total_power = simps(psd, dx=freq_res)
    delta_rel_power = delta_power / total_power
    tetha_rel_power = tetha_power / total_power
    alpha_rel_power = alpha_power / total_power
    beta_rel_power = beta_power / total_power
    gamma_rel_power = gamma_power / total_power
    
    delta_stage4.append(delta_rel_power)
    tetha_stage4.append(tetha_rel_power)
    alpha_stage4.append(alpha_rel_power)
    beta_stage4.append(beta_rel_power)
    gamma_stage4.append(gamma_rel_power)

In [ ]:
bands_stage4 = {
    "Delta" : delta_stage4,
    "Tetha" : tetha_stage4,
    "Alpha" : alpha_stage4,
    "Beta" : beta_stage4,
    "Gamma" : gamma_stage4
}

df_bands_stage4 = pd.DataFrame(bands_stage4, index=cols)

NameError: name 'delta_stage4' is not defined

In [ ]:
df_bands_stage4.head()

NameError: name 'df_bands_stage4' is not defined

In [ ]:
df_bands_stage4.plot(kind='bar', figsize=(20, 8))
plt.xlabel('Channels')
plt.ylabel('Relatives Power')
plt.show

NameError: name 'df_bands_stage4' is not defined

In [ ]:
# Mean Channles Values
mean_delta_stage4 = np.mean(delta_stage4)
mean_tetha_stage4 = np.mean(tetha_stage4)
mean_alpha_stage4 = np.mean(alpha_stage4)
mean_beta_stage4 = np.mean(beta_stage4)
mean_gamma_stage4 = np.mean(gamma_stage4)

# Data
categories = ["Delta", "Theta", "Alpha", "Beta", "Gamma"]
values = [mean_delta_stage4, mean_tetha_stage4, mean_alpha_stage4, mean_beta_stage4, mean_gamma_stage4]

# Plot
plt.figure(figsize=(8, 5))
plt.bar(categories, values, color=['blue', 'green', 'orange', 'red', 'purple'])

# Adding labels and title
plt.title("Mean Band Relative Power - VIAT-Map", fontsize=16)
plt.xlabel("EEG Bands", fontsize=12)
plt.ylabel("Relative Power", fontsize=12)
plt.xticks(fontsize=10)
plt.yticks(fontsize=10)
plt.tight_layout()
plt.show()

NameError: name 'delta_stage4' is not defined

# Combine Stage

In [ ]:
mean_delta_all = [mean_delta_stage1, mean_delta_stage2, mean_delta_stage3, mean_delta_stage4]
mean_tetha_all = [mean_tetha_stage1, mean_tetha_stage2, mean_tetha_stage3, mean_tetha_stage4]
mean_alpha_all = [mean_alpha_stage1, mean_alpha_stage2, mean_alpha_stage3, mean_alpha_stage4]
mean_beta_all = [mean_beta_stage1, mean_beta_stage2, mean_beta_stage3, mean_beta_stage4]
mean_gamma_all = [mean_gamma_stage1, mean_gamma_stage2, mean_gamma_stage3, mean_gamma_stage4]

NameError: name 'mean_delta_stage1' is not defined

In [ ]:
all_stages = {
    "Delta" : mean_delta_all,
    "Tetha" : mean_tetha_all,
    "Alpha" : mean_alpha_all,
    "Beta" : mean_beta_all,
    "Gamma" : mean_gamma_all
}

df_all_stages = pd.DataFrame(all_stages, index=["Resting", "Go/NoGo", "Reading", "VIAT-Map"])

NameError: name 'mean_delta_all' is not defined

In [ ]:
df_all_stages.plot(kind='bar', figsize=(20, 8), rot=0)
plt.xlabel('Stages')
plt.ylabel('Relatives Power')
plt.show

NameError: name 'df_all_stages' is not defined